In [ ]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones
)
from src.processor import (
    XPECProcessor
    
)
from src.utils.topos import getEstacionamientos

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas,
    getPlanificacionComercialDia,
    getCirculacionesComerciales)
import requests


In [ ]:

def hacerPeticion(method: str, URL: str, headers: dict = {}, data: dict = None):

    headers = {
        "Content-Type": "application/json; charset=UTF-8",
        "Prefer": "respond-async",
        **headers,
    }

    response = requests.request(method, URL, headers=headers, data=data)
    if response.status_code == 200:
        return response
    print("")
    if response.status_code >= 400:
        print(f"Error: '{response.status_code}' en la respuesta")
        return
    if response.status_code >= 300:
        print(f"Redirección: código'{response.status_code}'")
        return
    if response.status_code > 200:
        print(f"👍: código'{response.status_code}'")
        return
    if response.status_code < 200:
        print(f"Info: código'{response.status_code}'")
        return
    if not response.encoding == "utf-8":
        response.encoding = "utf-8"
    return response


In [ ]:
import json
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime("2025-09-17").strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

In [ ]:

def parse_launching_date(ld):
    """
    Convierte launchingDate a pd.Timestamp:
    - Lista/tupla [YYYY, M, D] o [YYYY, M, D, hh, mm, ss]
    - String o cualquier otro formato reconocido por pd.to_datetime
    - Devuelve NaT si no se puede convertir
    """
    try:
        # Caso: lista o tupla
        if isinstance(ld, (list, tuple)) and len(ld) >= 3:
            y, m, d = int(ld[0]), int(ld[1]), int(ld[2])
            if len(ld) >= 6:
                hh, mm, ss = int(ld[3]), int(ld[4]), int(ld[5])
                return pd.Timestamp(year=y, month=m, day=d, hour=hh, minute=mm, second=ss)
            return pd.Timestamp(year=y, month=m, day=d)
        # Caso: otros tipos -> pd.to_datetime intenta convertir
        return pd.to_datetime(ld, errors='coerce')
    except Exception:
        return pd.NaT


In [ ]:

def extract_steps(journey):
    """
    Extrae y transforma los steps de 'journey' a una lista de dicts ya tipados.
    """
    print(journey)
    steps = (journey or {}).get("steps") or []
    out = []
    for s in steps:
        out.append({
            "step": s.get("step"),
            "index": s.get("index"),
            "pointId": s.get("pointId"),
            "parkingTrack": s.get("parkingTrack"),
            "parkingTrackForDeparture": s.get("parkingTrackForDeparture"),
            "stationaryType": s.get("stationaryType"),
            "parity": s.get("parity"),
            "previous": s.get("previous"),
            "next": s.get("next"),
            "technicalStop": s.get("technicalStop"),
            "steps24h": s.get("steps24h"),
            "circulationMode": s.get("circulationMode"),
        })
    return out


In [ ]:

rows = []
for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))

    steps = (el.get("dayTrain") or {}).get("journey").get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"),        # renombrado a 'pointID' como pediste
            "Vía_Planificada": s.get("parkingTrack")
        })

planificacion = pd.DataFrame(rows)


In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        pro=pro,
        maniobra= maniobra
    )
    # historico = historico[
    #     (historico["Fecha"] >= pd.to_datetime(start_date))
    #     & (historico["Fecha"] <= pd.to_datetime(end_date))
    # ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2025-09-09"
end_date = "2025-09-10"
estaciones = []

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
historico_mes.sort_values(by=["Fecha"], inplace=True)

In [ ]:
ruta_planificación = Path(r"c:\Users\xiangzhou.zhang\Downloads\Calidad_2 (9).csv")
planificación= pd.read_csv(ruta_planificación, sep=",")

In [ ]:
estaciones = loadEstaciones()


In [ ]:
estaciones["Delegación"].unique()

In [ ]:
estaciones = estaciones[["Delegación", "Código"]]

In [ ]:

import gc

csv_folder = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\BI\Nueva carpeta")
files = sorted(csv_folder.glob("*.csv"))

# ajustar dtypes para reducir memoria (añade columnas que conozcas)
dtypes = {
    "Cod NUM_Tren": "str",
    "Fecha Origen Tren (YYYYMMDD) N": "str",
    "Via Estacionamiento":"str",

}

chunks = []
cols_keep = ["Desc Delegacion/Gerencia PR","Cod  PR","Fecha Origen Tren (YYYYMMDD) N", "Cod NUM_Tren", "Via Estacionamiento"]  
for f in files:
    try:
        reader = pd.read_csv(f, dtype=dtypes, encoding="utf-8", chunksize=200000, low_memory=True)
    except Exception:
        reader = pd.read_csv(f, dtype=dtypes, encoding="latin1", chunksize=200000, low_memory=True)
    for chunk in reader:
        keep = [c for c in cols_keep if c in chunk.columns]
        chunk = chunk[keep].copy()
        chunk.rename(columns={"Desc Delegacion/Gerencia PR":"Delegación","Cod  PR":"Código","Fecha Origen Tren (YYYYMMDD) N":"FechaOrigen","Cod NUM_Tren":"NTécnico"}, inplace=True)
        chunk["FechaOrigen"] = pd.to_datetime(chunk["FechaOrigen"], format="%Y%m%d", errors="coerce")
        if "NTécnico" in chunk.columns:
            chunk["NTécnico"] = chunk["NTécnico"].apply(rellenarId).astype("category")
        if "Via Estacionamiento" in chunk.columns:
            chunk["Via Estacionamiento"] = chunk["Via Estacionamiento"].astype("category")
        chunks.append(chunk)
    gc.collect()

if chunks:
    planificación = pd.concat(chunks, ignore_index=True)
else:
    planificación = pd.DataFrame()


In [ ]:
planificación[planificación["Código"] == "05481"]

In [ ]:
planificación_1 = planificación[planificación["Delegación"] == "RC NORTE"].copy()

In [ ]:
planificación_1.rename(columns={"Fecha Origen Tren (YYYYMMDD) N":"FechaOrigen","Cod NUM_Tren":"NTécnico"}, inplace=True)
planificación_1["FechaOrigen"] = pd.to_datetime(planificación_1["FechaOrigen"], format="%Y%m%d")
planificación_1["NTécnico"] = planificación_1["NTécnico"].apply(rellenarId)

In [ ]:
fname = Path(r"data/Subdirección.csv")

In [ ]:
subdirección = pd.read_csv(fname)

In [ ]:
planificacion = pd.merge(planificacion, subdirección, on="Código", how="left")

In [ ]:
planificacion["Subdirección"].unique()

In [ ]:
planificacion_norte = planificacion[planificacion["Subdirección"] == "RC NORTE"].copy()

In [ ]:
planificacion_1 = planificacion_norte[["NTécnico", "FechaOrigen", "Código", "Vía_Planificada"]].copy()

In [ ]:
centro_historico  = pd.merge(
    subdirección,
    historico_pro,
    on = "Código",
    how="right"
)

In [ ]:
norte = centro_historico[centro_historico["Subdirección"] == "RC NORTE"].copy()

In [ ]:
norte.columns

In [ ]:
historico_pro.columns

In [ ]:
historico_mes_planif = pd.merge(
    norte,
    planificacion_1,
    on = ["FechaOrigen", "NTécnico","Código"],
    how="left",
)

In [ ]:
test = historico_mes_planif.copy()

In [ ]:
test.columns

In [ ]:
mov = test['Movimiento'].astype(str).str.strip().str.lower()
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {'origen', 'salida'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['origen', 'salida'])].copy()
origen_test = out_a[out_a["Movimiento"] == "SALIDA"].copy() 
origen_test["Tipo circulación"] = 'Origen'
origen_test.reset_index(drop=True, inplace=True)


In [ ]:
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {'llegada', 'salida'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['llegada', 'salida'])].copy()
paso = out_a[out_a["Movimiento"] == "LLEGADA"].copy() 
paso["Tipo circulación"] = 'Paso'
paso.reset_index(drop=True, inplace=True)


In [ ]:
has_both = (
    test.assign(_mov=mov)
      .groupby(['FechaOrigen', 'Código', 'NTécnico'])['_mov']
      .transform(lambda s: {"llegada",'fin'}.issubset(set(s)))
)
out_a = test[has_both & mov.isin(['llegada', 'fin'])].copy()
fin = out_a[out_a["Movimiento"] == "LLEGADA"].copy() 
fin["Tipo circulación"] = 'Fin'
fin.reset_index(drop=True, inplace=True)


In [ ]:
origen_1 = origen_test[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]
paso_1 = paso[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]
fin_1 = fin[["CTC","FechaOrigen","LíneaComercial","Código","Nombre","NTécnico","Tipo circulación","Vía","Vía_Planificada"]]

In [ ]:
circulación_mes = pd.concat([origen_1,fin_1,paso_1],ignore_index = True)

circulación_mes["Vía_Planificada"] = circulación_mes["Vía_Planificada"].apply(
    lambda x: str(x) if pd.notnull(x) else x
)
circulación_mes["Vía"] = circulación_mes["Vía"].apply(
    lambda x: str(x) if pd.notnull(x) else x

)


In [ ]:
circulación_mes['Era la que tenía planificada?'] = circulación_mes.apply(
    lambda row: 'N/A' if pd.isna(row['Vía_Planificada']) else row['Vía'] == row['Vía_Planificada'],
    axis=1
)


In [ ]:
# circulación_mes['Se ha anticipado por CTC'] = circulación_mes.apply(
#     lambda row: True if row["FuenteVía"] == "CTC" else False,
#     axis=1
# )


In [ ]:
# circulación_mes['Se ha anticipado por SITRA'] = circulación_mes.apply(
#     lambda row: True if row["FuenteVía"] in ["SITRA_AUDITED", "SITRA_PROVIDED"] else False,
#     axis=1
# )



In [ ]:
# circulación_mes.drop(columns=["FuenteVía"], inplace=True)

In [ ]:
test2= historico_mes_planif.sort_values(by=["NTécnico","Fecha"]).copy()

In [ ]:
sub_dfs = [group for _, group in test2.groupby('NTécnico')]
dfs = []
for df in sub_dfs:
    Fecha1 = [group for _, group in df.groupby('FechaOrigen')]
    dfs.append(Fecha1)


In [ ]:
from itertools import chain

# dfs es una lista de listas (sublistas por NTécnico/FechaOrigin). Aplanamos de forma segura:
if hasattr(dfs, "flatten"):
    dddf = dfs.flatten()  # por si dfs fuera un ndarray
else:
    dddf = list(chain.from_iterable(dfs))

In [ ]:
ddfs = []
for df in dddf:
    codigo = [group for _, group in df.groupby('Código')]
    ddfs.append(codigo)

In [ ]:
información_adicional = []
for tren in ddfs:
    for elemento in tren:
        if isinstance(elemento, pd.DataFrame) and 'Movimiento' in elemento.columns:
            mask_aproximacion = elemento['Movimiento'].str.contains(
                r'APROXIMACIÓN|PREVISIÓN', 
                case=False, 
                na=False,
                regex=True
            )
            mask_llegada = elemento['Movimiento'].str.contains(
                r'LLEGADA', 
                case=False, 
                na=False,
                regex=True
            )
            mask_origen = elemento['Movimiento'].str.contains(
                r'ORIGEN', 
                case=False, 
                na=False,
                regex=True
            )
            mask_salida= elemento['Movimiento'].str.contains(
                r'SALIDA', 
                case=False, 
                na=False,
                regex=True
            )

            aproximacion = elemento[mask_aproximacion]
            llegada = elemento[mask_llegada]
            origen = elemento[mask_origen]
            salida = elemento[mask_salida]
            # print("aproximacion:",aproximacion)
            # print("llegada:",llegada)
            # print("origen:",origen)
            # print("salida:",salida)
            
            
            # Verificar primero si hay registros de origen
            if not origen.empty:
                print("origen")
                if origen["FuenteVía"].iloc[0] == "CTC":
                    print("CTC")
                    tiempo_prevision = origen["Fecha"].iloc[0]
                    NTécnico = origen["NTécnico"].iloc[0]
                    Código = origen["Código"].iloc[0]
                    print("NTécnico;",NTécnico)
                    Fecha = origen["FechaOrigen"].iloc[0]
                    # Buscar la llegada correspondiente
                    salida_correspondiente = salida
                    if not salida.empty:
                        tiempo_salida = salida["Fecha"].iloc[0]
                        tiempo_anticipación_CTC = tiempo_salida - tiempo_prevision
                    else:
                        tiempo_anticipación_CTC = "NA"
                    Sitra= False
                    Anticipación_sitra = "NA"
                    CTC = True

                    column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                    información_adicional.append(column)
                elif origen["FuenteVía"].iloc[0] in ["SITRA_AUDITED", "SITRA_PROVIDED"]:
                        print("Sitra")
                        tiempo_prevision = origen["Fecha"].iloc[0]
                        NTécnico = origen["NTécnico"].iloc[0]
                        Fecha = origen["FechaOrigen"].iloc[0]
                        Código = origen["Código"].iloc[0]
                        print("NTécnico;",NTécnico)
                        salida_correspondiente = salida
                        print("tiempo_Previson:",tiempo_prevision)
                        Sitra= True
                        if not salida.empty:
                            tiempo_salida = salida["Fecha"].iloc[0]
                            Anticipación_sitra = tiempo_salida - tiempo_prevision
                        else:
                            Anticipación_sitra = "NA"
                        CTC = False
                        tiempo_anticipación_CTC ="NA"
                        column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                        información_adicional.append(column)
            
            # Si no hay origen pero hay aproximación
            elif not aproximacion.empty:
                print("aproximación")
                if aproximacion["FuenteVía"].iloc[0] == "CTC":
                    print("CTC")
                    tiempo_prevision = aproximacion["Fecha"].iloc[0]
                    NTécnico = aproximacion["NTécnico"].iloc[0]
                    print("NTécnico;",NTécnico)
                    Fecha = aproximacion["FechaOrigen"].iloc[0]
                    Código = aproximacion["Código"].iloc[0]
                    # Buscar la llegada correspondiente
                    llegada_correspondiente = llegada
                    # print("llegada_correspondiente",llegada_correspondiente)
                    # print(llegada_correspondiente)
                    # print("tiempo_llegada:",tiempo_llegada)
                    print("tiempo_prevision",tiempo_prevision)
                    if not llegada.empty:
                        tiempo_llegada = llegada["Fecha"].iloc[0]
                        tiempo_anticipación_CTC = tiempo_llegada - tiempo_prevision
                    else:
                        tiempo_anticipación_CTC = "NA"
                    Sitra= False
                    Anticipación_sitra = "NA"
                    CTC = True

                    column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                    información_adicional.append(column)
                        
                
                elif aproximacion["FuenteVía"].iloc[0] in ["SITRA_AUDITED", "SITRA_PROVIDED"]:
                        print("Sitra")
                        tiempo_prevision = aproximacion["Fecha"].iloc[0]
                        NTécnico = aproximacion["NTécnico"].iloc[0]
                        print("NTécnico;",NTécnico)
                        Fecha = aproximacion["FechaOrigen"].iloc[0]
                        Código = aproximacion["Código"].iloc[0]
                        # Buscar la llegada correspondiente
                        llegada_correspondiente = llegada
                        Sitra= True
                        if not llegada.empty:
                            tiempo_llegada = llegada["Fecha"].iloc[0]
                            Anticipación_sitra = tiempo_llegada - tiempo_prevision
                        else:
                            Anticipación_sitra = "NA"
                        CTC = False
                        tiempo_anticipación_CTC = "NA"
                        column = {"NTécnico":NTécnico,"FechaOrigen":Fecha,"Código":Código,"Se ha anticipado por CTC":CTC,"Tiempo de anticipación CTC":tiempo_anticipación_CTC,"Se ha anticipado por Sitra":Sitra,"Tiempo de anticipación SITRA":Anticipación_sitra}
                        información_adicional.append(column) 

                                        
        else:
            print("El elemento no es un DataFrame o no tiene columna 'Movimiento'")


In [ ]:
df= pd.DataFrame(información_adicional)


In [ ]:
test = pd.merge(
    circulación_mes,
    df,
    on=["NTécnico","FechaOrigen","Código"],
    how = "left"
)

In [ ]:
test.rename(columns={"Vía":"Vía Estacionmamiento real","Via Estacionamiento":"Vía Estacionamiento planif"}, inplace=True)

In [ ]:
test

In [ ]:
def convertir_a_mm_ss(valor):
    if pd.isna(valor) or valor == "NA":
        return valor
    try:
        # Extraer la parte del tiempo (después de 'days ')
        tiempo = valor.split()[-1]
        horas, minutos, segundos = map(int, tiempo.split(':'))
        # Convertir todo a segundos y luego a minutos:segundos
        total_segundos = horas * 3600 + minutos * 60 + segundos
        mm = total_segundos // 60
        ss = total_segundos % 60
        return f"{mm:02d}:{ss:02d}"
    except:
        return valor  # En caso de que el formato no sea el esperado


In [ ]:
test["Tiempo de anticipación CTC"]= test['Tiempo de anticipación CTC'].apply(convertir_a_mm_ss)


In [ ]:
test["Tiempo de anticipación SITRA"]= test['Tiempo de anticipación SITRA'].apply(convertir_a_mm_ss)

In [ ]:
test

In [ ]:
test["Tiempo de anticipación CTC"] = test["Tiempo de anticipación CTC"].astype(str)
test["Tiempo de anticipación SITRA"] = test["Tiempo de anticipación SITRA"].astype(str)

In [ ]:
# test.rename(columns={"Tiempo de anticipación CTC":"Tiempo de anticipación CTC(mm:ss)","Tiempo de anticipación SITRA":"Tiempo de anticipación SITRA(mm:ss)"}, inplace=True)

In [ ]:
test 

<h1>Resumen diario</h1>


In [ ]:
diario = test.copy()

In [ ]:
conteo = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
conteo

In [ ]:
estaciones = loadEstaciones()

In [ ]:
estaciones = estaciones[["CTC","Código"]].copy()

In [ ]:
merge = pd.merge(
    estaciones,
    conteo,
    on=["Código"],
    how = "right"
)

In [ ]:
diario.head(4)

In [ ]:
conteo_1 = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
conteo_1

In [ ]:
coincide = conteo_1[conteo_1["Era la que tenía planificada?"] == True]

In [ ]:
NoConcide = conteo_1[conteo_1["Era la que tenía planificada?"] == False]

In [ ]:
NoConcide.head(4)

In [ ]:
merge.rename(columns={"count":"Número total de circulaciones que han llegado realmente a esa vía "}, inplace=True)

In [ ]:
merge.head(4)

In [ ]:
merge_1 = pd.merge(
    merge,
    NoConcide,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1 = pd.merge(
    merge_1,
    coincide,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1

In [ ]:
merge_1.rename(columns={"count_x":"Número de circulaciones que han llegado a esa vía y no la tenían planificada","count_y":"Número de circulaciones que han llegado a esa vía y si la tenían planificada"}, inplace=True)

In [ ]:
merge_1["Número de circulaciones que han llegado a esa vía y no la tenían planificada"] = merge_1.apply(
    lambda row: 0 if  pd.isna(row["Número de circulaciones que han llegado a esa vía y no la tenían planificada"])else row["Número de circulaciones que han llegado a esa vía y no la tenían planificada"], axis=1)


In [ ]:
merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"] = merge_1.apply(lambda row: 0 if  pd.isna(
    row["Número de circulaciones que han llegado a esa vía y si la tenían planificada"])else row["Número de circulaciones que han llegado a esa vía y si la tenían planificada"], axis=1)


In [ ]:
merge_1

In [ ]:
sitra = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?","Se ha anticipado por Sitra"]).size().reset_index(name='count')

In [ ]:
anticipación_sitra = sitra[sitra["Se ha anticipado por Sitra"]].copy()

In [ ]:
merge_1 = pd.merge(
    merge_1,
    anticipación_sitra,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1.drop(columns=["Se ha anticipado por Sitra"], inplace=True)

In [ ]:
merge_1.rename(columns={"count":"Número de circulaciones con Anticipación SITRA"}, inplace=True)

In [ ]:
merge_1["Número de circulaciones con Anticipación SITRA"] = merge_1["Número de circulaciones con Anticipación SITRA"].apply(
    lambda x: 0 if pd.isna(x) else x
)

In [ ]:
merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"] = merge_1["Número de circulaciones que han llegado a esa vía y si la tenían planificada"].apply(
    lambda x: 0 if pd.isna(x) else x)


In [ ]:
tiempo_sitra  = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Tiempo de anticipación SITRA","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
anticipación_sitra = tiempo_sitra[~tiempo_sitra["Tiempo de anticipación SITRA"].isin(["NA","nan"])].copy()

In [ ]:
anticipación_sitra.drop(columns=["count"], inplace=True)

In [ ]:
sub_dfs = [group for _, group in anticipación_sitra.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real"])]


In [ ]:
for df in sub_dfs:
    df['Tiempo de anticipación SITRA'] = pd.to_timedelta(df['Tiempo de anticipación SITRA'])
    # print(df['Tiempo de anticipación SITRA'])
    df['Tiempo de antelación de anuncio SITRA'] = df['Tiempo de anticipación SITRA'].mean()
    df['Tiempo de antelación de anuncio SITRA'] = df['Tiempo de antelación de anuncio SITRA'].apply(lambda x: f"{int(x.total_seconds() // 60):02}:{int(x.total_seconds() % 60):02}")


In [ ]:
df_completo = pd.concat(sub_dfs, ignore_index=True)

In [ ]:
df_completo.drop(columns=["Tiempo de anticipación SITRA"], inplace=True)

In [ ]:
df_completo

In [ ]:
merge_1 = pd.merge(
    merge_1,
    df_completo,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1[~merge_1["Tiempo de antelación de anuncio SITRA"].isna()]

In [ ]:
merge_1["Tiempo de antelación de anuncio SITRA"] = merge_1["Tiempo de antelación de anuncio SITRA"].apply(
    lambda x: 0 if pd.isna(x) else x)

In [ ]:
merge_1[~merge_1["Tiempo de antelación de anuncio SITRA"].isin([0])]

In [ ]:
ctc = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?","Se ha anticipado por CTC"]).size().reset_index(name='Número total de circulaciones con estimación de vía auditada CTC')

In [ ]:
anticipación_ctc= ctc[ctc["Se ha anticipado por CTC"]].copy()

In [ ]:
anticipación_ctc

In [ ]:
merge_1 = pd.merge( 
    merge_1,
    anticipación_ctc,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1.drop(columns=["Se ha anticipado por CTC"], inplace=True)

In [ ]:
tiempo_ctc  = diario.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Tiempo de anticipación CTC","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
anticipación_ctc = tiempo_ctc[~tiempo_ctc["Tiempo de anticipación CTC"].isin(["NA","nan"])].copy()

In [ ]:
anticipación_ctc.drop(columns=["count"], inplace=True)

In [ ]:
anticipación_ctc

In [ ]:
sub_dfs = [group for _, group in anticipación_ctc.groupby(["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"])]


In [ ]:
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    # print(df['Tiempo de anticipación SITRA'])
    df['Tiempo de antelación de anuncio CTC'] = df['Tiempo de anticipación CTC'].mean()
    df['Tiempo de antelación de anuncio CTC'] = df['Tiempo de antelación de anuncio CTC'].apply(lambda x: f"{int(x.total_seconds() // 60):02}:{int(x.total_seconds() % 60):02}")

In [ ]:
df_completo = pd.concat(sub_dfs, ignore_index=True) 

In [ ]:
df_completo.drop(columns=["Tiempo de anticipación CTC"], inplace=True)

In [ ]:
df_completo.drop_duplicates(subset=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"], inplace=True)

In [ ]:
merge_1 = pd.merge(
    merge_1,
    df_completo,
    on=["FechaOrigen","Código","Nombre","Tipo circulación","Vía Estacionmamiento real","Era la que tenía planificada?"],
    how = "left"
)

In [ ]:
merge_1

In [ ]:
data={"detalle_tren_mensual":test,
      "resumen_diario":merge_1,
}

In [ ]:
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\rc_noroeste_semana.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
circulación = test.groupby(["FechaOrigen","Tipo circulación","Nombre"]).size().reset_index(name='Número total de circulación')

In [ ]:
circulación["Tipo circulación"].unique()

In [ ]:
circulación_origen = circulación[circulación["Tipo circulación"] == "Origen"].copy()

In [ ]:
anticipación_origen = test[test["Tipo circulación"] == "Origen"].copy()
anticipación_origen = anticipación_origen[anticipación_origen["Se ha anticipado por CTC"] == True].copy()

In [ ]:
anticipación_origen['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_origen['Tiempo de anticipación CTC'])

In [ ]:
anticipación_origen =anticipación_origen[anticipación_origen["Tiempo de anticipación CTC"] < timedelta(minutes=25)]

In [ ]:
sub_dfs = [group for _, group in anticipación_origen.groupby(["FechaOrigen","Nombre"])]


In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        "Nombre": df['Nombre'].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
    })


In [ ]:
tiempo_origen = pd.DataFrame(tiempo_medio)


In [ ]:
circulación_origen.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_origen = test[test["Tipo circulación"] == "Origen"].copy()

In [ ]:
fiabilidad_origen_1 = fiabilidad_origen.groupby(["FechaOrigen","Era la que tenía planificada?","Nombre"]).size().reset_index(name='count')

In [ ]:
fiabilidad_origen_1

In [ ]:
sub_dfs = [group for _, group in fiabilidad_origen_1.groupby(["FechaOrigen","Nombre"])]

In [ ]:
sub_dfs[0]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        "Nombre": df['Nombre'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado

    })
    


In [ ]:
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
circulación_origen["FechaOrigen"] = circulación_origen["FechaOrigen"].astype(str)
tiempo_origen["FechaOrigen"] = tiempo_origen["FechaOrigen"].astype(str)
estadistica = pd.merge(
    circulación_origen,
    tiempo_origen,
    on=["FechaOrigen","Nombre"],
    how = "left"
)

In [ ]:
porcentaje_fiabilidad["FechaOrigen"]= porcentaje_fiabilidad["FechaOrigen"].astype(str)
estadistica = pd.merge(
    estadistica,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Nombre"],
    how="left"
)

In [ ]:
estadistica

In [ ]:
circulación_destino = circulación[circulación["Tipo circulación"] == "Fin"].copy()
anticipación_destino = test[test["Tipo circulación"] == "Fin"].copy()
anticipación_destino = anticipación_destino[anticipación_destino["Se ha anticipado por CTC"] == True].copy()
anticipación_destino['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_destino['Tiempo de anticipación CTC'])
anticipación_destino =anticipación_destino[anticipación_destino["Tiempo de anticipación CTC"] < timedelta(minutes=25)]
sub_dfs = [group for _, group in anticipación_destino.groupby(["FechaOrigen","Nombre"])]

In [ ]:
circulación_destino

In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        'Nombre':df['Nombre'].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
        
    })


In [ ]:
tiempo_destino = pd.DataFrame(tiempo_medio)

In [ ]:
tiempo_medio

In [ ]:
circulación_destino.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_destino = test[test["Tipo circulación"] == "Fin"].copy()
fiabilidad_destino_1 = fiabilidad_destino.groupby(["FechaOrigen","Era la que tenía planificada?","Nombre"]).size().reset_index(name='count')

In [ ]:
sub_dfs = [group for _, group in fiabilidad_destino_1.groupby(["FechaOrigen","Nombre"])]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        'Nombre':df['Nombre'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado,
    })
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
circulación_destino

In [ ]:
circulación_destino["FechaOrigen"] = circulación_destino["FechaOrigen"].astype(str)
tiempo_destino["FechaOrigen"] = tiempo_destino["FechaOrigen"].astype(str)
estadistica_destino = pd.merge(
    circulación_destino,
    tiempo_destino,
    on=["FechaOrigen","Nombre"],
    how = "left"
)

In [ ]:
porcentaje_fiabilidad["FechaOrigen"] = porcentaje_fiabilidad["FechaOrigen"].astype(str)
estadistica_destino = pd.merge(
    estadistica_destino,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Nombre"],
    how="left"
)

In [ ]:
estadistica_destino

In [ ]:
circulación_Paso = circulación[circulación["Tipo circulación"] == "Paso"].copy()
anticipación_Paso = test[test["Tipo circulación"] == "Paso"].copy()
anticipación_Paso = anticipación_Paso[anticipación_Paso["Se ha anticipado por CTC"] == True].copy()
anticipación_Paso['Tiempo de anticipación CTC'] = pd.to_timedelta(anticipación_Paso['Tiempo de anticipación CTC'])
anticipación_Paso =anticipación_Paso[anticipación_Paso["Tiempo de anticipación CTC"] < timedelta(minutes=25)]
sub_dfs = [group for _, group in anticipación_Paso.groupby(["FechaOrigen","Nombre"])]

In [ ]:
tiempo_medio = []
for df in sub_dfs:
    df['Tiempo de anticipación CTC'] = pd.to_timedelta(df['Tiempo de anticipación CTC'])
    fecha = df['FechaOrigen'].iloc[0]
    tiempo_promedio = df['Tiempo de anticipación CTC'].mean()
    tiempo_promedio = f"{int(tiempo_promedio.total_seconds() // 60):02}:{int(tiempo_promedio.total_seconds() % 60):02}"

    tiempo_medio.append({
        'FechaOrigen': fecha,
        'Nombre':df["Nombre"].iloc[0],
        'Tiempo medio anticipación CTC': tiempo_promedio
    })
tiempo_paso = pd.DataFrame(tiempo_medio)

In [ ]:
circulación_Paso.sort_values(by=["FechaOrigen"], inplace=True)

In [ ]:
fiabilidad_paso = test[test["Tipo circulación"] == "Paso"].copy()
fiabilidad_paso_1 = fiabilidad_paso.groupby(["FechaOrigen","Nombre","Era la que tenía planificada?"]).size().reset_index(name='count')

In [ ]:
sub_dfs = [group for _, group in fiabilidad_paso_1.groupby(["FechaOrigen","Nombre"])]

In [ ]:
fiabilidades = []
for df in sub_dfs:
    df['count'] = df['count'].astype(int)
    fecha = df['FechaOrigen'].iloc[0]
    total = df['count'].sum()
    if total > 0:
        porcentaje_planificado = (df[df['Era la que tenía planificada?'] == True]['count'].sum() / total) * 100
        porcentaje_no_planificado = (df[df['Era la que tenía planificada?'] == False]['count'].sum() / total) * 100
    else:
        porcentaje_planificado = 0
        porcentaje_no_planificado = 0
    porcentaje_planificado = round(porcentaje_planificado, 2)
    porcentaje_no_planificado = round(porcentaje_no_planificado, 2)

    fiabilidades.append({
        'FechaOrigen': fecha,
        'Nombre':df['Nombre'].iloc[0],
        'Fiabilidad vía planificada(%)': porcentaje_planificado,
    })
porcentaje_fiabilidad = pd.DataFrame(fiabilidades)

In [ ]:
circulación_Paso["FechaOrigen"] = circulación_Paso["FechaOrigen"].astype(str)
tiempo_paso["FechaOrigen"] = tiempo_paso["FechaOrigen"].astype(str)
estadistica_paso = pd.merge(
    circulación_Paso,
    tiempo_paso,
    on=["FechaOrigen","Nombre"],
    how = "left"
)

In [ ]:
porcentaje_fiabilidad["FechaOrigen"] = porcentaje_fiabilidad["FechaOrigen"].astype(str)
estadistica_paso = pd.merge(
    estadistica_paso,
    porcentaje_fiabilidad,
    on=["FechaOrigen","Nombre"],
    how="left"
)

In [ ]:
estadistica_total = pd.concat(
    [estadistica, estadistica_destino,estadistica_paso], ignore_index=True
)

In [ ]:
estadistica_total

In [ ]:
test["FechaOrigen"] = test["FechaOrigen"].astype(str)
merge_1["FechaOrigen"] = merge_1["FechaOrigen"].astype(str)
data = {
    "estadistica": estadistica_total,
    "detalle_tren_mensual":test,
    "resumen_diario":merge_1,
    
}
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\RC_Norte_11_09.xlsx")

In [ ]:
guardarExcelMulti(data, fname)

In [ ]:
a = planificacion_norte[planificacion_norte["NTécnico"] == "04085"]

In [ ]:
a[a["Código"] == "13103"]

In [ ]:
a[a["Código"] == "13102"]

In [ ]:
a[a["Código"] == "13104"]

In [ ]:
test_1 = test[test["FechaOrigen"] == "2025-09-09"]

In [ ]:
x =test_1[test_1["Nombre"] == "LUIAONDO (APD)"]

In [ ]:
x[x["NTécnico"] == "26572"]

In [ ]:
[test["Nombre"] == "LUIAONDO (APD)"].head(10)

In [ ]:
test[test["Nombre"] == "SANTA CRUZ DE LLODIO (APD)"].head(10)

In [ ]:
test[test["Nombre"] == "SALBIO (APD)"].head(10)

In [ ]:
test[test["Nombre"] == "AMURRIO"].head(10)

In [ ]:
test[test["Nombre"] == "AMURRIO IPARRALDE (APD)"].head(10)